# Text-To-Sign Production Colab

This is the single supported project-wide Colab notebook.

Current public stage: Dataset Build.

Implemented internal downstream surface: Baseline Modeling.

Not yet implemented: broader evaluation, contribution modeling, playback/demo.

The notebook is orchestration-only. Dataset and modeling logic stays in repository modules under `src/`. From Dataset Build outputs onward, each major step is split into separate reuse, extract, and run/publish cells.


## Section 0: Runtime And Session Setup

This section defines run settings and prepares the Colab runtime tools.


### 0.1 Runtime Settings

What this step does: defines the repository checkout target, Dataset Build split list, baseline run name, and qualitative panel size.

Required inputs: none; edit only these constants before running later cells.

Produced outputs: notebook session variables used by later cells.

When this step may be skipped: never in a fresh runtime; later cells depend on these settings.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")
RUNTIME_ROOT = WORKTREE_ROOT.parent

PIPELINE_SPLITS = ["train", "val", "test"]
BASELINE_RUN_NAME = "baseline-default"
BASELINE_PANEL_SIZE = 8

print(f"Python: {sys.version.split()[0]}")
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Worktree root: {WORKTREE_ROOT}")
print(f"Dataset Build splits: {PIPELINE_SPLITS}")
print(f"Baseline run name: {BASELINE_RUN_NAME}")
print(f"Baseline panel size: {BASELINE_PANEL_SIZE}")

### 0.2 Runtime Tool Setup

What this step does: ensures the Colab runtime has `zstd`, which is required for `.tar.zst` archive extraction and creation.

Required inputs: a Colab runtime with package installation permission.

Produced outputs: an available `zstd` executable.

When this step may be skipped: only when `zstd` is already available; the cell detects that and exits cleanly.


In [ ]:
if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
    print("Installed zstd.")
else:
    print("zstd is already available.")

## Section 1: Drive Mount

This section connects the fixed Google Drive storage layout used by the supported workflow.


### 1.1 Mount Google Drive

What this step does: mounts Google Drive at `/content/drive`.

Required inputs: a Google account with the fixed project Drive layout.

Produced outputs: mounted Drive paths for raw inputs and published artifacts.

When this step may be skipped: only if Drive is already mounted in the current runtime.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Drive mounted at /content/drive")
print("Dataset Build raw input root: /content/drive/MyDrive/text-to-sign-production/raw/how2sign")
print(
    "Dataset Build artifact root: "
    "/content/drive/MyDrive/text-to-sign-production/artifacts/dataset-build/processed-v1/"
)
print(
    "Baseline Modeling run root: "
    f"/content/drive/MyDrive/text-to-sign-production/artifacts/baseline-modeling/runs/{BASELINE_RUN_NAME}/"
)

## Section 2: Repo Acquisition And Install

This section acquires the repository and installs the dependencies needed for Dataset Build and Baseline Modeling.


### 2.1 Acquire Repository

What this step does: clones the repository into `/content/text-to-sign-production` and checks out `REPO_REF`.

Required inputs: network access to the repository.

Produced outputs: a clean worktree at `WORKTREE_ROOT`.

When this step may be skipped: only if the desired repository revision is already checked out in the current runtime.


In [ ]:
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
try:
    current_cwd = Path.cwd()
except FileNotFoundError:
    current_cwd = None

if current_cwd is None or current_cwd == WORKTREE_ROOT or WORKTREE_ROOT in current_cwd.parents:
    os.chdir(RUNTIME_ROOT)

if WORKTREE_ROOT.exists():
    shutil.rmtree(WORKTREE_ROOT)

subprocess.run(["git", "clone", REPO_URL, str(WORKTREE_ROOT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=WORKTREE_ROOT, check=True)
print(f"Repository cloned at {WORKTREE_ROOT}")

### 2.2 Install Repository Requirements

What this step does: installs the Colab requirements, sets the working directory, and exposes `src/` on the Python import path.

Required inputs: a cloned worktree from the previous step.

Produced outputs: importable repository modules and printed git revision.

When this step may be skipped: only if the same worktree and dependencies are already prepared in the current runtime.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
    cwd=WORKTREE_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    cwd=WORKTREE_ROOT,
    check=True,
)
os.chdir(WORKTREE_ROOT)

if str(WORKTREE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WORKTREE_ROOT / "src"))

current_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Checked out revision: {current_revision}")
print("Python import path ready for direct library calls.")

### 2.3 Load Shared Workflow Helpers

What this step does: imports shared workflow constants and defines archive extraction helpers used by Dataset Build and Baseline Modeling cells.

Required inputs: installed repository dependencies and importable `src/` modules.

Produced outputs: shared constants plus `merge_extracted_tree` and `extract_baseline_archive` helper functions.

When this step may be skipped: never when using any Dataset Build or Baseline Modeling reuse, extract, run, or inspection cell.


In [ ]:
from text_to_sign_production.data.constants import (
    COLAB_DRIVE_ARTIFACTS_ROOT,
    MANIFESTS_AND_REPORTS_ARCHIVE_NAME,
    SAMPLE_ARCHIVE_NAME_TEMPLATE,
)
from text_to_sign_production.ops.archive_ops import extract_tar_zst_with_progress


def merge_extracted_tree(source_root: Path, destination_root: Path) -> None:
    for source in source_root.iterdir():
        destination = destination_root / source.name
        if source.is_dir():
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)


def _resolve_baseline_run_root(target_run_root: Path | None = None) -> Path:
    if target_run_root is not None:
        return target_run_root

    if "BASELINE_LAYOUT" not in globals():
        raise RuntimeError(
            "BASELINE_LAYOUT is not defined. Run the baseline prepare cell first or pass "
            "target_run_root explicitly."
        )

    baseline_layout = globals()["BASELINE_LAYOUT"]
    run_root = getattr(baseline_layout, "run_root", None)
    if run_root is None:
        raise RuntimeError(
            "BASELINE_LAYOUT.run_root is not available. Re-run the baseline prepare cell "
            "or pass target_run_root explicitly."
        )

    return Path(run_root)


def extract_baseline_archive(archive_path: Path, label: str, target_run_root: Path | None = None) -> None:
    run_root = _resolve_baseline_run_root(target_run_root)
    restore_root = Path(tempfile.mkdtemp(prefix=f"{archive_path.name}.restore-", dir="/content"))
    try:
        extracted_root = restore_root / "extracted"
        extract_tar_zst_with_progress(archive_path, extracted_root, label=label)
        merge_extracted_tree(extracted_root, run_root)
    finally:
        shutil.rmtree(restore_root, ignore_errors=True)


print("Shared workflow helpers loaded.")

## Section 3: Dataset Build Outputs

This section resolves Dataset Build outputs by reuse, extraction, or run/publish.


### 3.1 Reuse Extracted Dataset Build Outputs

What this step does: checks whether processed Dataset Build manifests and sample directories already exist in the worktree.

Required inputs: extracted outputs under `data/processed/v1/`.

Produced outputs: the boolean `DATASET_BUILD_EXTRACTED_READY`.

When this step may be skipped: never before Baseline Modeling; use it to decide whether extraction or a fresh run is needed.


In [ ]:
def dataset_build_outputs_ready() -> bool:
    manifests_ready = all(
        (WORKTREE_ROOT / "data/processed/v1/manifests" / f"{split}.jsonl").is_file()
        for split in PIPELINE_SPLITS
    )
    samples_ready = all(
        (WORKTREE_ROOT / "data/processed/v1/samples" / split).is_dir()
        and any((WORKTREE_ROOT / "data/processed/v1/samples" / split).glob("*.npz"))
        for split in PIPELINE_SPLITS
    )
    return manifests_ready and samples_ready


DATASET_BUILD_EXTRACTED_READY = dataset_build_outputs_ready()
print(f"Dataset Build extracted outputs ready: {DATASET_BUILD_EXTRACTED_READY}")
for split in PIPELINE_SPLITS:
    print(WORKTREE_ROOT / "data/processed/v1/manifests" / f"{split}.jsonl")
    print(WORKTREE_ROOT / "data/processed/v1/samples" / split)

### 3.2 Extract Archived Dataset Build Outputs

What this step does: restores Dataset Build extracted outputs from the fixed Drive artifact archives.

Required inputs: `dataset_build_manifests_reports.tar.zst` plus `dataset_build_samples_<split>.tar.zst` archives under the fixed Drive artifact root.

Produced outputs: extracted manifests, reports, and samples under `data/interim/` and `data/processed/v1/`.

When this step may be skipped: skip when the reuse cell reports extracted outputs are ready, or when no Dataset Build archives are available and a fresh run is needed.


In [ ]:
if dataset_build_outputs_ready():
    print("Dataset Build extracted outputs already exist; extraction skipped.")
else:
    archive_paths = [COLAB_DRIVE_ARTIFACTS_ROOT / MANIFESTS_AND_REPORTS_ARCHIVE_NAME]
    archive_paths.extend(
        COLAB_DRIVE_ARTIFACTS_ROOT / SAMPLE_ARCHIVE_NAME_TEMPLATE.format(split=split)
        for split in PIPELINE_SPLITS
    )
    missing_archives = [path for path in archive_paths if not path.is_file()]
    if missing_archives:
        raise FileNotFoundError(
            "Dataset Build archives are not available; run the next "
            "Dataset Build run/publish cell. "
            + "Missing: "
            + ", ".join(path.as_posix() for path in missing_archives)
        )

    restore_root = Path(tempfile.mkdtemp(prefix="dataset-build-restore-", dir="/content"))
    try:
        for archive_path in archive_paths:
            destination = restore_root / archive_path.name.replace(".tar.zst", "")
            extract_tar_zst_with_progress(
                archive_path,
                destination,
                label=f"[dataset-build] Extract {archive_path.name}",
            )
            merge_extracted_tree(destination, WORKTREE_ROOT)
        print("Dataset Build archives extracted into the worktree.")
    finally:
        shutil.rmtree(restore_root, ignore_errors=True)

DATASET_BUILD_EXTRACTED_READY = dataset_build_outputs_ready()
print(f"Dataset Build extracted outputs ready: {DATASET_BUILD_EXTRACTED_READY}")

### 3.3 Run Dataset Build And Publish Outputs

What this step does: runs Dataset Build from fixed Drive raw inputs and publishes Dataset Build archives to the fixed Drive artifact root.

Required inputs: fixed Drive raw translations and keypoint archives.

Produced outputs: extracted Dataset Build outputs in the worktree and archived outputs in Drive.

When this step may be skipped: skip when extracted outputs are already ready; run the extract cell first when archived outputs are available.


In [ ]:
from text_to_sign_production.data.constants import (
    COLAB_DRIVE_ARTIFACTS_ROOT,
    DEFAULT_FILTER_CONFIG_RELATIVE_PATH,
    MANIFESTS_AND_REPORTS_ARCHIVE_NAME,
    SAMPLE_ARCHIVE_NAME_TEMPLATE,
)
from text_to_sign_production.workflows.dataset_build import run_dataset_build

if dataset_build_outputs_ready():
    print("Dataset Build extracted outputs already exist; run/publish skipped.")
else:
    dataset_archives = [COLAB_DRIVE_ARTIFACTS_ROOT / MANIFESTS_AND_REPORTS_ARCHIVE_NAME]
    dataset_archives.extend(
        COLAB_DRIVE_ARTIFACTS_ROOT / SAMPLE_ARCHIVE_NAME_TEMPLATE.format(split=split)
        for split in PIPELINE_SPLITS
    )
    existing_dataset_archives = [path for path in dataset_archives if path.is_file()]
    if existing_dataset_archives:
        if len(existing_dataset_archives) == len(dataset_archives):
            raise RuntimeError(
                "Dataset Build archives already exist. Run the Dataset Build "
                "extract cell before starting a fresh run."
            )
        raise RuntimeError(
            "A partial Dataset Build archive set already exists in the Drive "
            "artifact root. Clean up the published Dataset Build archives or "
            "run the Dataset Build extract cell before starting a fresh run. "
            f"Existing archives: {', '.join(str(path) for path in existing_dataset_archives)}"
        )

    dataset_result = run_dataset_build(
        splits=tuple(PIPELINE_SPLITS),
        filter_config_path=WORKTREE_ROOT / DEFAULT_FILTER_CONFIG_RELATIVE_PATH,
        input_mode="fixed_colab_drive",
        output_mode="fixed_colab_drive",
    )
    print(f"Dataset Build completed for splits: {', '.join(dataset_result.splits)}")
    for path in dataset_result.output_paths:
        print(f"published Dataset Build archive: {path}")

DATASET_BUILD_EXTRACTED_READY = dataset_build_outputs_ready()
print(f"Dataset Build extracted outputs ready: {DATASET_BUILD_EXTRACTED_READY}")

## Section 4: Baseline Modeling Training Outputs

This section prepares the baseline run root, then resolves training outputs by reuse, extraction, or run/publish.


### 4.1 Prepare Baseline Run Root

What this step does: validates processed Dataset Build inputs and writes baseline config files into the run root.

Required inputs: processed Dataset Build train and validation outputs.

Produced outputs: `config/baseline.yaml`, `config/source_baseline.yaml`, and `BASELINE_LAYOUT`.

When this step may be skipped: skip only if the same run root and config are already prepared in this runtime.


In [ ]:
from text_to_sign_production.modeling.config import DEFAULT_BASELINE_CONFIG_PATH
from text_to_sign_production.workflows.baseline_modeling import (
    COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    run_baseline_modeling,
)

BASELINE_CONFIG_PATH = DEFAULT_BASELINE_CONFIG_PATH

prepare_result = run_baseline_modeling(
    step="prepare",
    config_path=BASELINE_CONFIG_PATH,
    run_name=BASELINE_RUN_NAME,
    artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    panel_size=BASELINE_PANEL_SIZE,
)
BASELINE_LAYOUT = prepare_result.layout

print(f"Prepared baseline run root: {BASELINE_LAYOUT.run_root}")
for step in prepare_result.steps:
    print(f"{step.step}: {step.action} -> {step.output_path}")

### 4.2 Reuse Extracted Training Outputs

What this step does: checks whether training checkpoints and run summaries already exist under the run root.

Required inputs: extracted `checkpoints/` and `metrics/` outputs.

Produced outputs: the boolean `TRAINING_EXTRACTED_READY`.

When this step may be skipped: never before qualitative export; use it to decide whether extraction or training is needed.


In [ ]:
def training_outputs_ready() -> bool:
    return all(
        path.is_file()
        for path in (
            BASELINE_LAYOUT.config_path,
            BASELINE_LAYOUT.source_config_path,
            BASELINE_LAYOUT.checkpoint_run_summary_path,
            BASELINE_LAYOUT.metrics_run_summary_path,
            BASELINE_LAYOUT.last_checkpoint_path,
        )
    )


TRAINING_EXTRACTED_READY = training_outputs_ready()
print(f"Training extracted outputs ready: {TRAINING_EXTRACTED_READY}")
print(f"Training archive: {BASELINE_LAYOUT.training_archive_path}")

### 4.3 Extract Archived Training Outputs

What this step does: restores `config/`, `checkpoints/`, and `metrics/` from `baseline_training_outputs.tar.zst`.

Required inputs: `archives/baseline_training_outputs.tar.zst` under the run root.

Produced outputs: extracted baseline training outputs under the run root.

When this step may be skipped: skip when training extracted outputs already exist, or when no training archive exists and a fresh training run is needed.


In [ ]:
if training_outputs_ready():
    print("Training extracted outputs already exist; extraction skipped.")
elif BASELINE_LAYOUT.training_archive_path.is_file():
    extract_baseline_archive(
        BASELINE_LAYOUT.training_archive_path,
        "[baseline train] Extract archived outputs",
    )
    print("Training archive extracted.")
else:
    raise FileNotFoundError("Training archive is not available; run the training run/publish cell.")

TRAINING_EXTRACTED_READY = training_outputs_ready()
print(f"Training extracted outputs ready: {TRAINING_EXTRACTED_READY}")

### 4.4 Run Training And Publish Outputs

What this step does: runs baseline training and writes extracted training outputs plus `baseline_training_outputs.tar.zst`.

Required inputs: prepared run root and processed Dataset Build train/validation outputs.

Produced outputs: `checkpoints/`, `metrics/`, and `archives/baseline_training_outputs.tar.zst`.

When this step may be skipped: skip when extracted training outputs already exist; run the extract cell first when the training archive exists.


In [ ]:
if training_outputs_ready():
    print("Training extracted outputs already exist; run/publish skipped.")
elif BASELINE_LAYOUT.training_archive_path.is_file():
    raise RuntimeError("Training archive exists. Run the training extract cell before a fresh run.")
else:
    training_result = run_baseline_modeling(
        step="train",
        config_path=BASELINE_CONFIG_PATH,
        run_name=BASELINE_RUN_NAME,
        artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
        panel_size=BASELINE_PANEL_SIZE,
    )
    BASELINE_LAYOUT = training_result.layout
    for step in training_result.steps:
        print(f"{step.step}: {step.action} -> {step.output_path}")
        if step.archive_path is not None:
            print(f"archive: {step.archive_path}")

TRAINING_EXTRACTED_READY = training_outputs_ready()
print(f"Training extracted outputs ready: {TRAINING_EXTRACTED_READY}")

## Section 5: Qualitative Panel Outputs

This section resolves qualitative panel outputs by reuse, extraction, or run/publish.


### 5.1 Reuse Extracted Qualitative Panel Outputs

What this step does: checks whether qualitative panel outputs already exist under the run root.

Required inputs: extracted `qualitative/` outputs.

Produced outputs: the boolean `QUALITATIVE_EXTRACTED_READY`.

When this step may be skipped: never before record/package assembly; use it to decide whether extraction or export is needed.


In [ ]:
def qualitative_outputs_ready() -> bool:
    return all(
        path.is_file()
        for path in (
            BASELINE_LAYOUT.panel_definition_path,
            BASELINE_LAYOUT.qualitative_records_path,
            BASELINE_LAYOUT.panel_summary_path,
            BASELINE_LAYOUT.evidence_bundle_path,
        )
    )


QUALITATIVE_EXTRACTED_READY = qualitative_outputs_ready()
print(f"Qualitative extracted outputs ready: {QUALITATIVE_EXTRACTED_READY}")
print(f"Qualitative archive: {BASELINE_LAYOUT.qualitative_archive_path}")

### 5.2 Extract Archived Qualitative Panel Outputs

What this step does: restores `qualitative/` from `baseline_qualitative_outputs.tar.zst`.

Required inputs: `archives/baseline_qualitative_outputs.tar.zst` under the run root.

Produced outputs: extracted qualitative panel outputs under `qualitative/`.

When this step may be skipped: skip when qualitative extracted outputs already exist, or when no qualitative archive exists and a fresh panel export is needed.


In [ ]:
if qualitative_outputs_ready():
    print("Qualitative extracted outputs already exist; extraction skipped.")
elif BASELINE_LAYOUT.qualitative_archive_path.is_file():
    extract_baseline_archive(
        BASELINE_LAYOUT.qualitative_archive_path,
        "[baseline qualitative] Extract archived outputs",
    )
    print("Qualitative archive extracted.")
else:
    raise FileNotFoundError(
        "Qualitative archive is not available; run the qualitative run/publish cell."
    )

QUALITATIVE_EXTRACTED_READY = qualitative_outputs_ready()
print(f"Qualitative extracted outputs ready: {QUALITATIVE_EXTRACTED_READY}")

### 5.3 Run Qualitative Panel Export And Publish Outputs

What this step does: exports the validation qualitative panel and writes extracted qualitative outputs plus `baseline_qualitative_outputs.tar.zst`.

Required inputs: extracted training outputs, including a usable checkpoint and `run_summary.json`.

Produced outputs: `qualitative/` and `archives/baseline_qualitative_outputs.tar.zst`.

When this step may be skipped: skip when extracted qualitative outputs already exist; run the extract cell first when the qualitative archive exists.


In [ ]:
if qualitative_outputs_ready():
    print("Qualitative extracted outputs already exist; run/publish skipped.")
elif BASELINE_LAYOUT.qualitative_archive_path.is_file():
    raise RuntimeError(
        "Qualitative archive exists. Run the qualitative extract cell before a fresh export."
    )
elif not training_outputs_ready():
    raise RuntimeError("Training outputs are required before qualitative export.")
else:
    qualitative_result = run_baseline_modeling(
        step="export-panel",
        config_path=BASELINE_CONFIG_PATH,
        run_name=BASELINE_RUN_NAME,
        artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
        panel_size=BASELINE_PANEL_SIZE,
    )
    BASELINE_LAYOUT = qualitative_result.layout
    for step in qualitative_result.steps:
        print(f"{step.step}: {step.action} -> {step.output_path}")
        if step.archive_path is not None:
            print(f"archive: {step.archive_path}")

QUALITATIVE_EXTRACTED_READY = qualitative_outputs_ready()
print(f"Qualitative extracted outputs ready: {QUALITATIVE_EXTRACTED_READY}")

## Section 6: Record/Package Outputs

This section resolves runtime record/package outputs by reuse, extraction, or assemble/package and publish.


### 6.1 Reuse Extracted Record/Package Outputs

What this step does: checks whether runtime record/package outputs already exist under the run root.

Required inputs: extracted `record/` outputs.

Produced outputs: the boolean `RECORD_PACKAGE_EXTRACTED_READY`.

When this step may be skipped: skip only if final inspection is not needed.


In [ ]:
def record_package_outputs_ready() -> bool:
    return all(
        path.is_file()
        for path in (
            BASELINE_LAYOUT.package_manifest_path,
            BASELINE_LAYOUT.record_evidence_bundle_path,
            BASELINE_LAYOUT.record_run_summary_path,
        )
    )


RECORD_PACKAGE_EXTRACTED_READY = record_package_outputs_ready()
print(f"Record/package extracted outputs ready: {RECORD_PACKAGE_EXTRACTED_READY}")
print(f"Record archive: {BASELINE_LAYOUT.record_archive_path}")

### 6.2 Extract Archived Record/Package Outputs

What this step does: restores `record/` from `baseline_record_package.tar.zst`.

Required inputs: `archives/baseline_record_package.tar.zst` under the run root.

Produced outputs: extracted record/package outputs under `record/`.

When this step may be skipped: skip when record/package extracted outputs already exist, or when no record archive exists and assembly is needed.


In [ ]:
if record_package_outputs_ready():
    print("Record/package extracted outputs already exist; extraction skipped.")
elif BASELINE_LAYOUT.record_archive_path.is_file():
    extract_baseline_archive(
        BASELINE_LAYOUT.record_archive_path,
        "[baseline record] Extract archived package",
    )
    print("Record/package archive extracted.")
else:
    raise FileNotFoundError(
        "Record/package archive is not available; run the assemble/package cell."
    )

RECORD_PACKAGE_EXTRACTED_READY = record_package_outputs_ready()
print(f"Record/package extracted outputs ready: {RECORD_PACKAGE_EXTRACTED_READY}")

### 6.3 Assemble/Package And Publish Record Outputs

What this step does: assembles the runtime-side baseline record package and writes `baseline_record_package.tar.zst`.

Required inputs: extracted training outputs and extracted qualitative panel outputs.

Produced outputs: `record/` and `archives/baseline_record_package.tar.zst`.

When this step may be skipped: skip when record/package outputs already exist; run the extract cell first when the record archive exists.


In [ ]:
if record_package_outputs_ready():
    print("Record/package extracted outputs already exist; assemble/package skipped.")
elif BASELINE_LAYOUT.record_archive_path.is_file():
    raise RuntimeError("Record/package archive exists. Run the record/package extract cell first.")
elif not qualitative_outputs_ready():
    raise RuntimeError("Qualitative outputs are required before record/package assembly.")
else:
    package_result = run_baseline_modeling(
        step="package",
        config_path=BASELINE_CONFIG_PATH,
        run_name=BASELINE_RUN_NAME,
        artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
        panel_size=BASELINE_PANEL_SIZE,
    )
    BASELINE_LAYOUT = package_result.layout
    for step in package_result.steps:
        print(f"{step.step}: {step.action} -> {step.output_path}")
        if step.archive_path is not None:
            print(f"archive: {step.archive_path}")
    print(f"Baseline package manifest: {package_result.layout.package_manifest_path}")

RECORD_PACKAGE_EXTRACTED_READY = record_package_outputs_ready()
print(f"Record/package extracted outputs ready: {RECORD_PACKAGE_EXTRACTED_READY}")

## Section 7: Final Artifact Inspection

This section prints the resolved Dataset Build and Baseline Modeling artifact surfaces for copy-back into experiment records.


### 7.1 Inspect Final Artifact Paths

What this step does: prints the key extracted outputs, archived outputs, and formal record inputs.

Required inputs: any completed Dataset Build or Baseline Modeling outputs from prior sections.

Produced outputs: a text summary of paths to cite in records or handoff notes.

When this step may be skipped: skip only when no final artifact summary is needed.


In [ ]:
from text_to_sign_production.data.constants import COLAB_DRIVE_ARTIFACTS_ROOT

print("Current public stage: Dataset Build")
print("Implemented internal downstream surface: Baseline Modeling")
print("Not yet implemented: broader evaluation, contribution modeling, playback/demo")
print()
print(f"Dataset Build extracted ready: {dataset_build_outputs_ready()}")
print(f"Dataset Build artifact root: {COLAB_DRIVE_ARTIFACTS_ROOT}")
print(f"Baseline run root: {BASELINE_LAYOUT.run_root}")
print(f"Training extracted ready: {training_outputs_ready()}")
print(f"Qualitative extracted ready: {qualitative_outputs_ready()}")
print(f"Record/package extracted ready: {record_package_outputs_ready()}")
print()
print("Baseline archive paths:")
print(BASELINE_LAYOUT.training_archive_path)
print(BASELINE_LAYOUT.qualitative_archive_path)
print(BASELINE_LAYOUT.record_archive_path)
print()
print(
    "Formal Sprint 3 baseline records should cite run_summary.json, last.pt/best.pt, "
    "qualitative outputs, baseline_evidence_bundle.json, "
    "baseline_modeling_package.json, and these archives."
)